In [1]:
import fastf1.core
import fastf1.plotting
import pandas as pd
import numpy as np
from sqlalchemy.dialects.sqlite import insert
from sqlalchemy import select
from backend.database.db_init import ENGINE
from sqlalchemy.orm import sessionmaker
from backend.database.models import EventSession, EventRound, SessionName, Weather, RaceControl, Lap, Driver
#from backend.modules.db_tools import get_data_for_event_session, insert_for_weather, get_race_control_messages, get_lap_data
from IPython.display import clear_output
import fastf1
import os
from dotenv import load_dotenv
load_dotenv()

Session = sessionmaker(bind=ENGINE)

year = 2025
gp = 1

session_data = fastf1.get_session(year, gp, 5)
session_data.load()

req         WARNING 	DEFAULT CACHE ENABLED! (32.54 GB) /home/kurios/.cache/fastf1
core           INFO 	Loading data for Australian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weat

In [ ]:
def get_event_session(session, year, round_number, session_name):
    stmt_event_session = (
        select(EventSession)
            .join(EventRound)
            .join(SessionName)
            .filter(EventRound.year == year)
            .filter(EventRound.round_number == round_number)
            .filter(SessionName.name == session_name)
    )
    return session.execute(stmt_event_session).scalars().all()

def get_list_drivers(session, results_event_session):
    stmt_drivers = (
        select(Lap)
            .join(Driver)
            .join(EventSession)
            .filter(EventSession.id == results_event_session[0].id)
    )
    return session.execute(stmt_drivers).scalars().unique()

def get_lap_per_driver(session, results_event_session, driver):
    stmt_driver = (
    select(Lap)
        .join(Driver)
        .join(EventSession)
        .filter(EventSession.id == results_event_session[0].id)
        .filter(Driver.number == driver)
    )
    return session.execute(stmt_driver).scalars().unique()

def format_car_data(telemetry):
    df = telemetry.copy()
    
    try:
        df = df.add_distance().add_differential_distance().add_relative_distance().add_track_status()
    except Exception:
        pass

    df['time_ms'] = (df['Time'].dt.total_seconds() * 1000).astype(int)
    df['date'] = pd.to_datetime(df['Date']).dt.date
    
    df['isBraking'] = df['Brake'].astype(bool)
    df['rpm'] = df['RPM'].fillna(0).astype(int)
    df['speed'] = df['Speed'].fillna(0).astype(int)
    df['throttle'] = df['Throttle'].fillna(0).astype(int)
    df['track_status'] = df['TrackStatus'].fillna(0).astype(int)
    
    mapping = {
        'rpm': 'rpm', 'speed': 'speed', 'nGear': 'gear', 'throttle': 'throttle',
        'isBraking': 'isBraking', 'time_ms': 'time_ms', 'Distance': 'distance',
        'DifferentialDistance': 'differential_distance', 'RelativeDistance': 'relative_distance',
        'TrackStatus': 'track_status'
    }
    
    existing_cols = [c for c in mapping.keys() if c in df.columns]
    return df[existing_cols].rename(columns=mapping)

def format_pos_data(telemetry):
    df = telemetry.copy()
    
    try:
        df = df.add_track_status()
    except Exception:
        pass

    df['Time'] = df['Time'].apply(lambda x: int(x.total_seconds()*1000) if pd.notna(x) else None)
    df['Date'] = pd.to_datetime(df['Date']).dt.date

    df['SessionTime'] = pd.to_timedelta(df['SessionTime'])
    df['SessionTime'] = df['SessionTime'].apply(lambda x: x.to_pytimedelta() if pd.notna(x) else None)

    df = df.replace({pd.NaT: None, np.nan: None}).copy()
    int_cols = ['X', 'Y', 'Z', 'Time', 'TrackStatus']
    for col in int_cols:
        df[col] = df[col].apply(lambda x: int(x) if pd.notna(x) else 0)

    mapping = {
            'X': 'x',
            'Y': 'y',
            'Z': 'z',
            'Time': 'time_ms',
            'SessionTime': 'session_time',
            'Date': 'date',
            'Status': 'is_on_track',
            'TrackStatus': 'track_status'
            }

    existing_cols = [c for c in mapping.keys() if c in df.columns]
    return df[existing_cols].rename(columns=mapping)

In [ ]:
laps = session_data.laps

all_drivers_car_data = []
all_drivers_pos_data = []

with Session() as session:
    results_event_session = get_event_session(session, 2025, 5, 'Race')
    results_drivers = get_list_drivers(session, results_event_session)
    list_drivers = list(set([ld.driver.number for ld in results_drivers]))

    for driver_num in list_drivers:

        driver_laps = laps.pick_drivers(driver_num)

        if driver_laps.empty:
            print(f"Skipping Driver {driver_num}: Not found in FastF1 laps data.")
            continue

        results = get_lap_per_driver(session, results_event_session, driver_num)
        
        for lap_obj in results:
            lap_slice = driver_laps.pick_laps(lap_obj.lap_number)

            if lap_slice.empty:
                print(
                    f"Skipping Driver {driver_num} Lap {lap_obj.lap_number}: "
                    "Lap not present in FastF1."
                )
                continue

            try:
                raw_car_slice = lap_slice.get_car_data()
                raw_pos_slice = lap_slice.get_pos_data()
            except ValueError as e:
                print(
                    f"Skipping Driver {driver_num} Lap {lap_obj.lap_number}: {e}"
                )
                continue

            if raw_car_slice.empty or raw_pos_slice.empty:
                print(
                    f"Skipping Driver {driver_num} Lap {lap_obj.lap_number}: "
                    "No telemetry recorded."
                )
                continue

            car_data = format_car_data(raw_car_slice) 
            pos_data = format_pos_data(raw_pos_slice)
            
            car_data['lap_id'] = lap_obj.id
            pos_data['lap_id'] = lap_obj.id

            all_drivers_car_data.append(car_data)
            all_drivers_pos_data.append(pos_data)

if all_drivers_car_data:
    final_car_df = pd.concat(all_drivers_car_data, ignore_index=True)
    
if all_drivers_pos_data:
    final_pos_df = pd.concat(all_drivers_pos_data, ignore_index=True)


2025-12-27 23:17:52,169 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-12-27 23:17:52,171 INFO sqlalchemy.engine.Engine SELECT event_session.id, event_session.date, event_session.event_round_id, event_session.session_name_id 
FROM event_session JOIN event_round ON event_round.id = event_session.event_round_id JOIN session_name ON session_name.id = event_session.session_name_id 
WHERE event_round.year = ? AND event_round.round_number = ? AND session_name.name = ?
2025-12-27 23:17:52,171 INFO sqlalchemy.engine.Engine [cached since 343.2s ago] (2025, 5, 'Race')
2025-12-27 23:17:52,173 INFO sqlalchemy.engine.Engine SELECT lap.id, lap.laptime_ms, lap.lap_number, lap.sector_1_ms, lap.sector_2_ms, lap.sector_3_ms, lap.stint, lap.speed_i1, lap.speed_i2, lap.speed_fl, lap.speed_st, lap.tyre_life, lap.position, lap.sector_1_time, lap.sector_2_time, lap.sector_3_time, lap.pit_in_time_ms, lap.pit_out_time_ms, lap.start_time, lap.start_date, lap.is_personal_best, lap.is_deleted, lap.is_accurat

In [21]:
final_car_df

,rpm,speed,gear,throttle,isBraking,time_ms,distance,differential_distance,relative_distance,track_status,lap_id
0,10043,0,2,16,True,119,0.000000,0.000000,0.000000,1,1563
1,9752,0,2,16,True,398,0.000000,0.000000,0.000000,1,1563
2,8072,0,2,16,False,638,0.000000,0.000000,0.000000,1,1563
3,6392,8,2,16,False,878,0.533333,0.533333,0.000101,1,1563
4,4992,13,2,16,False,1078,1.255556,0.722222,0.000238,1,1563
...,...,...,...,...,...,...,...,...,...,...,...
294966,10869,275,7,100,False,94997,5168.965556,15.201389,0.985189,4,2162
294967,10959,277,7,100,False,95318,5193.664722,24.699167,0.989896,4,2162
294968,11077,279,7,100,False,95598,5215.364722,21.700000,0.994032,4,2162
294969,11148,281,7,100,False,95838,5234.098056,18.733333,0.997603,4,2162


In [22]:
final_pos_df

,x,y,z,time_ms,session_time,date,is_on_track,track_status,lap_id
0,-941,-1576,86,194,0 days 01:11:00.549000,2025-03-16,OnTrack,1,1563
1,-941,-1576,86,574,0 days 01:11:00.929000,2025-03-16,OnTrack,1,1563
2,-943,-1573,86,774,0 days 01:11:01.129000,2025-03-16,OnTrack,1,1563
3,-951,-1567,86,1174,0 days 01:11:01.529000,2025-03-16,OnTrack,1,1563
4,-956,-1562,86,1334,0 days 01:11:01.689000,2025-03-16,OnTrack,1,1563
...,...,...,...,...,...,...,...,...,...
301728,-412,-2077,90,94740,0 days 02:43:03.689000,2025-03-16,OnTrack,4,2162
301729,-643,-1858,88,95020,0 days 02:43:03.969000,2025-03-16,OnTrack,4,2162
301730,-799,-1710,88,95440,0 days 02:43:04.389000,2025-03-16,OnTrack,4,2162
301731,-929,-1587,86,95620,0 days 02:43:04.569000,2025-03-16,OnTrack,4,2162
